# Figure 6: scIDiff Multiome Validation - User-Friendly Guide\n\nThis notebook demonstrates how to use **scIDiff** to analyze multiome data (scRNA-seq + scATAC-seq) and generate Figure 6 from the manuscript.\n\n## What This Notebook Does\n\n1. Loads and preprocesses E18 mouse brain multiome data\n2. Trains a **Hybrid Drift Field** with velocity priors using scIDiff\n3. Extracts **regulatory archetypes** from Jacobian decomposition\n4. Integrates **ATAC-seq** data to validate chromatin accessibility dynamics\n5. Generates **Figure 6** showing the alignment between regulatory archetypes and chromatin accessibility\n\n## Before You Start\n\nMake sure you have installed:\n```bash\npip install scanpy torch scikit-learn matplotlib pandas numpy scipy\npip install -e /path/to/scIDiff_V2\n```

## Configuration\n\n**Update these paths to point to your data files:**

In [ ]:
# ========== USER CONFIGURATION ==========\n# Update these paths to match your data location\n\nPATH_TO_SCRNA_DATA = '/home/ubuntu/upload/filtered_feature_bc_matrix'\nPATH_TO_SCATAC_DATA = '/home/ubuntu/upload/e18_mouse_brain_fresh_5k_atac_fragments.tsv'\nOUTPUT_DIR = '/home/ubuntu/figure6_analysis'\n\n# Model hyperparameters (you can adjust these)\nN_PCA_DIMS = 30  # Number of PCA dimensions to use\nN_EPOCHS = 50    # Number of training epochs\nN_ARCHETYPES = 4  # Number of regulatory archetypes to extract\n\nprint('Configuration set!')

## Step 1: Import Libraries

In [ ]:
import numpy as np\nimport pandas as pd\nimport scanpy as sc\nimport torch\nimport matplotlib.pyplot as plt\nimport matplotlib.gridspec as gridspec\nfrom scipy.stats import spearmanr\nfrom sklearn.neighbors import NearestNeighbors\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Import scIDiff\nfrom scqdiff.models.drift import DriftField, DriftConfig\n\n# Set random seeds\nnp.random.seed(42)\ntorch.manual_seed(42)\n\nprint('✓ Libraries imported successfully')

## Step 2: Load and Preprocess Data

In [ ]:
print('Loading data...')\n\n# Load 10X multiome data\nadata = sc.read_10x_mtx(PATH_TO_SCRNA_DATA, var_names='gene_symbols', cache=True)\n\n# Filter for Gene Expression only (exclude ATAC peaks)\ngene_mask = ~adata.var_names.str.startswith('chr')\nadata = adata[:, gene_mask].copy()\n\nprint(f'Loaded {adata.n_obs} cells × {adata.n_vars} genes')\n\n# Quality control\nsc.pp.filter_cells(adata, min_genes=200)\nsc.pp.filter_genes(adata, min_cells=3)\nadata.var['mt'] = adata.var_names.str.startswith('mt-')\nsc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)\nadata = adata[adata.obs.n_genes_by_counts < 6000, :]\nadata = adata[adata.obs.pct_counts_mt < 20, :].copy()\n\nprint(f'After QC: {adata.n_obs} cells × {adata.n_vars} genes')\n\n# Normalize and scale\nsc.pp.normalize_total(adata, target_sum=1e4)\nsc.pp.log1p(adata)\nsc.pp.highly_variable_genes(adata, n_top_genes=2000)\nadata.raw = adata\nsc.pp.scale(adata, max_value=10)\n\nprint('✓ Data preprocessing complete')

## Step 3: Dimensionality Reduction and Trajectory Inference

In [ ]:
# PCA\nsc.tl.pca(adata, n_comps=50)\n\n# Neighbors and UMAP\nsc.pp.neighbors(adata, n_pcs=N_PCA_DIMS)\nsc.tl.umap(adata)\nsc.tl.leiden(adata, resolution=0.5)\n\n# Pseudotime\nadata.uns['iroot'] = np.argmax(adata.obsm['X_pca'][:, 0])\nsc.tl.diffmap(adata)\nsc.tl.dpt(adata)\n\nprint(f'✓ Identified {len(adata.obs["leiden"].unique())} clusters')\n\n# Visualize\nfig, axes = plt.subplots(1, 2, figsize=(12, 4))\nsc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Cell Clusters')\nsc.pl.umap(adata, color='dpt_pseudotime', ax=axes[1], show=False, title='Pseudotime')\nplt.tight_layout()\nplt.show()

## Step 4: Estimate Velocity Prior\n\nWe use a simplified pseudotime-based velocity approximation. For real analysis, use scVelo.

In [ ]:
X_pca = adata.obsm['X_pca'][:, :N_PCA_DIMS]\nn_cells, n_dims = X_pca.shape\n\n# Compute velocity using k-nearest neighbors\nnbrs = NearestNeighbors(n_neighbors=30).fit(X_pca)\ndistances, indices = nbrs.kneighbors(X_pca)\n\nV_pca = np.zeros_like(X_pca)\nW_confidence = np.zeros(n_cells)\n\nfor i in range(n_cells):\n    neighbors = indices[i, 1:]\n    neighbor_pts = adata.obs['dpt_pseudotime'].values[neighbors]\n    current_pt = adata.obs['dpt_pseudotime'].values[i]\n    \n    forward_mask = neighbor_pts > current_pt\n    if forward_mask.sum() > 0:\n        forward_neighbors = neighbors[forward_mask]\n        V_pca[i] = (X_pca[forward_neighbors] - X_pca[i]).mean(axis=0)\n        W_confidence[i] = forward_mask.sum() / len(neighbors)\n    else:\n        V_pca[i] = (X_pca[neighbors] - X_pca[i]).mean(axis=0)\n        W_confidence[i] = 0.1\n\n# Normalize\nV_norm = np.linalg.norm(V_pca, axis=1, keepdims=True)\nV_pca = V_pca / (V_norm + 1e-8)\n\nprint(f'✓ Velocity estimated (mean confidence: {W_confidence.mean():.3f})')

## Step 5: Train scIDiff Hybrid Drift Field

In [ ]:
# Convert to PyTorch tensors\nX_torch = torch.FloatTensor(X_pca)\nV_torch = torch.FloatTensor(V_pca)\nW_torch = torch.FloatTensor(W_confidence)\nT_torch = torch.FloatTensor(adata.obs['dpt_pseudotime'].values)\n\n# Configure model with velocity prior\ncfg = DriftConfig(\n    dim=n_dims,\n    hidden=128,\n    depth=3,\n    beta=0.1,\n    use_velocity_prior=True,\n    vel_k=16,\n    vel_scale=0.5,\n    vel_tau=0.1,\n    vel_conf_power=2.0,\n    vel_schedule='mid'\n)\n\n# Initialize model\nmodel = DriftField(cfg, X_ref=X_torch, V_ref=V_torch, W_ref=W_torch)\noptimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n\n# Training loop\nprint(f'Training for {N_EPOCHS} epochs...')\nmodel.train()\nfor epoch in range(N_EPOCHS):\n    idx = np.random.choice(n_cells, 256, replace=False)\n    x_batch = X_torch[idx]\n    t_batch = torch.rand(256)\n    \n    optimizer.zero_grad()\n    drift = model(x_batch, t_batch)\n    loss = torch.nn.functional.mse_loss(drift, torch.zeros_like(drift))\n    loss.backward()\n    optimizer.step()\n    \n    if (epoch + 1) % 10 == 0:\n        print(f'  Epoch {epoch+1}/{N_EPOCHS}, Loss: {loss.item():.4f}')\n\nprint('✓ Training complete')

## Step 6: Compute Jacobian and Extract Archetypes

In [ ]:
model.eval()\n\n# Compute Jacobian at multiple time points\nn_time_points = 50\ntime_points = np.linspace(0, 1, n_time_points)\njacobians = []\n\nprint('Computing Jacobian tensor...')\nfor t_val in time_points:\n    # Sample cells near this pseudotime\n    pt_dist = np.abs(adata.obs['dpt_pseudotime'].values - t_val)\n    pt_mask = np.argsort(pt_dist)[:50]\n    \n    x_sample = X_torch[pt_mask]\n    t_sample = torch.full((len(x_sample),), t_val)\n    \n    # Compute Jacobian\n    x_sample.requires_grad_(True)\n    drift = model(x_sample, t_sample)\n    \n    jac_list = []\n    for i in range(len(x_sample)):\n        jac_row = []\n        for j in range(n_dims):\n            grad = torch.autograd.grad(drift[i, j], x_sample, retain_graph=True)[0]\n            jac_row.append(grad[i].detach().numpy())\n        jac_list.append(np.stack(jac_row))\n    \n    jacobians.append(np.mean(jac_list, axis=0))\n\njacobian_tensor = np.stack(jacobians)\nprint(f'✓ Jacobian shape: {jacobian_tensor.shape}')\n\n# Decompose into archetypes\nJ_matrix = jacobian_tensor.reshape(n_time_points, -1)\nU, S, Vt = np.linalg.svd(J_matrix, full_matrices=False)\narchetype_activations = U[:, :N_ARCHETYPES]\n\nprint(f'✓ Extracted {N_ARCHETYPES} archetypes')\n\n# Compute unstable mode scores\nunstable_scores = np.zeros(n_cells)\nfor i in range(n_cells):\n    t_val = adata.obs['dpt_pseudotime'].values[i]\n    t_idx = int(t_val * (n_time_points - 1))\n    J = jacobian_tensor[t_idx]\n    eigvals = np.linalg.eigvals(J)\n    unstable_scores[i] = np.sum(np.maximum(eigvals.real, 0))\n\nunstable_scores = (unstable_scores - unstable_scores.min()) / (unstable_scores.max() - unstable_scores.min() + 1e-10)\nadata.obs['unstable_mode_score'] = unstable_scores\n\n# Map archetypes to cells\nfor k in range(N_ARCHETYPES):\n    adata.obs[f'archetype_{k+1}'] = np.interp(adata.obs['dpt_pseudotime'].values, time_points, archetype_activations[:, k])

## Step 7: Load ATAC-seq Data

In [ ]:
print('Loading ATAC fragment data (this may take a few minutes)...')\n\natac_counts = {}\ntarget_barcodes = set(adata.obs_names)\n\nwith open(PATH_TO_SCATAC_DATA, 'r') as f:\n    for line in f:\n        if line.startswith('#'):\n            continue\n        parts = line.strip().split('\\t')\n        if len(parts) >= 5:\n            barcode = parts[3]\n            if barcode in target_barcodes:\n                count = int(parts[4])\n                atac_counts[barcode] = atac_counts.get(barcode, 0) + count\n\n# Add to adata\nadata.obs['atac_fragments'] = 0\nfor barcode, count in atac_counts.items():\n    if barcode in adata.obs_names:\n        adata.obs.loc[barcode, 'atac_fragments'] = count\n\nprint(f'✓ ATAC data loaded for {(adata.obs["atac_fragments"] > 0).sum()} cells')\nprint(f'  Mean fragments: {adata.obs["atac_fragments"].mean():.1f}')\n\n# Compute correlation\ncells_with_atac = adata.obs['atac_fragments'] > 0\nrho, pval = spearmanr(adata.obs.loc[cells_with_atac, 'unstable_mode_score'], adata.obs.loc[cells_with_atac, 'atac_fragments'])\nprint(f'  Correlation: ρ = {rho:.3f}, p = {pval:.2e}')

## Step 8: Generate Figure 6

In [ ]:
fig = plt.figure(figsize=(20, 12), dpi=600)\ngs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.4)\n\n# Panels a-d: Archetype spatial activation\nfor i in range(4):\n    ax = fig.add_subplot(gs[0, i])\n    sc.pl.umap(adata, color=f'archetype_{i+1}', cmap='viridis', ax=ax, show=False, title=f'Archetype {i+1}', frameon=False)\n\n# Panel e: ATAC vs unstable mode\nax_e = fig.add_subplot(gs[1, :2])\nhexbin = ax_e.hexbin(adata.obs.loc[cells_with_atac, 'unstable_mode_score'], adata.obs.loc[cells_with_atac, 'atac_fragments'], gridsize=30, cmap='Blues', bins='log')\nplt.colorbar(hexbin, ax=ax_e, label='Cell count (log)')\nax_e.set_xlabel('Unstable Mode Score', fontweight='bold')\nax_e.set_ylabel('ATAC Fragment Count', fontweight='bold')\nax_e.set_title(f'e) ATAC Accessibility vs Dynamical Instability\\nρ = {rho:.3f}, p = {pval:.2e}', fontweight='bold', loc='left')\n\n# Panel f: Temporal coordination\nax_f1 = fig.add_subplot(gs[1, 2:])\ncolors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3']\nfor i in range(4):\n    ax_f1.plot(time_points, archetype_activations[:, i], label=f'Archetype {i+1}', color=colors[i], linewidth=2)\nax_f1.set_xlabel('Pseudotime', fontweight='bold')\nax_f1.set_ylabel('Archetype Activation', fontweight='bold')\nax_f1.set_title('f) Temporal Coordination', fontweight='bold', loc='left')\nax_f1.legend()\n\n# Panel f2: Mean ATAC\nax_f2 = fig.add_subplot(gs[2, 2:], sharex=ax_f1)\npt_bins = np.linspace(0, 1, 51)\nmean_atac = []\npt_centers = []\nfor i in range(50):\n    bin_mask = (adata.obs['dpt_pseudotime'] >= pt_bins[i]) & (adata.obs['dpt_pseudotime'] < pt_bins[i+1])\n    if bin_mask.sum() > 0:\n        mean_atac.append(adata.obs.loc[bin_mask, 'atac_fragments'].mean())\n        pt_centers.append((pt_bins[i] + pt_bins[i+1]) / 2)\nax_f2.plot(pt_centers, mean_atac, color='darkred', linewidth=2)\nax_f2.fill_between(pt_centers, mean_atac, alpha=0.3, color='darkred')\nax_f2.set_xlabel('Pseudotime', fontweight='bold')\nax_f2.set_ylabel('Mean ATAC Fragments', fontweight='bold')\n\nfig.suptitle('Figure 6: Chromatin Accessibility Dynamics Align with Inferred Regulatory Archetypes', fontsize=14, fontweight='bold', y=0.995)\nplt.tight_layout()\n\n# Save\nfig.savefig(f'{OUTPUT_DIR}/Figure6.png', dpi=600, bbox_inches='tight')\nfig.savefig(f'{OUTPUT_DIR}/Figure6.pdf', dpi=600, bbox_inches='tight')\nplt.show()\n\nprint('✓ Figure 6 generated and saved!')

## Step 9: Save Results

In [ ]:
# Save processed data\nadata.write(f'{OUTPUT_DIR}/e18_scidiff_results.h5ad')\n\n# Save archetype profiles\narchetype_df = pd.DataFrame(archetype_activations, columns=[f'Archetype_{i+1}' for i in range(N_ARCHETYPES)])\narchetype_df['pseudotime'] = time_points\narchetype_df.to_csv(f'{OUTPUT_DIR}/archetype_profiles.csv', index=False)\n\nprint('✓ All results saved!')\nprint(f'\\nOutput files in {OUTPUT_DIR}:')\nprint('  - Figure6.png / Figure6.pdf')\nprint('  - e18_scidiff_results.h5ad')\nprint('  - archetype_profiles.csv')